# Alberta export potential with gravity-informed ML

I use this notebook as the walkthrough for the project package. It mirrors `Q1a.py` and `Q1b.py` so you can follow the framework without reading every helper.

Structural gravity + XGBoost on seven-economy HS4 trade; Alberta scored as its own exporter. See **Questions answered** below for the research questions this package addresses.

## Questions answered

**Question 1(a).** Using appropriate data, identify Alberta’s top ten export destination countries in 2025, ranked by total domestic export value. For each destination, report the top five HS4 products (code, name, and value).

**Question 1(b).** Develop a machine-learning model on 2024 bilateral HS4 trade among the United States, Canada, Mexico, China, South Korea, Japan, and the United Kingdom. Treat Alberta as a separate economic entity. Report predicted top ten destinations and top five HS4 products each. Distinguish observed 2025 exports (Q1a) from model-estimated export **potential** (Q1b).

Alberta potential is not the same as observed shipments. I calibrate dollar levels to Alberta’s observed 2025 total for presentation; ranks and product mix come from the model.

**Default mode.** `REFIT = False` loads saved tables so the notebook stays light for Git. You can set `REFIT = True` to re-run the scripts.


## Framework at a glance

```text
Q1a  CIMT domestic exports 2025 (AB)
     → roll HS6 to HS4, sum months, US states → country
     → observed top 10 destinations + top 5 HS4

Q1b  BACI 2024 HS4 trade among USA, CAN, MEX, CHN, KOR, JPN, GBR
     + Gravity / WDI / WITS features
     → Model 1 → Model 2 → Model 3  (pair holdout + GroupKFold)
     → score Alberta as its own exporter with Model 3
     → potential top 10 / top 5  (calibrated to observed total)
```


## The three Q1b models

| Model | Name | Learner | What I put in |
| --- | --- | --- | --- |
| **1** | **Structural gravity (GDP & trade costs)** | HistGBM | Exporter/importer GDP, distance, contiguity, common language, FTA. No product info. |
| **2** | **Product structure & MFN tariffs** | XGBoost | Model 1 + importer income & growth, HS2 / energy / agri flags, Balassa RCA, importer MFN. |
| **3** | **Market demand & exporter capability** | XGBoost | Model 2 + leave-one-out HS4 market size, demand growth, trade complementarity, complexity proxy, exporter capability. |

I score Alberta potential with **Model 3**. Models 1 and 2 are the validity ladder.

For partners outside the seven-economy training set, I proxy Model 3 importer HS4 market size as world HS4 market × (importer GDP / US GDP).

I summarise other learners (Random Forest, CatBoost, ensembles, openness / HHI trials) in `methodology/APPENDIX_other_models.md`. I do not fit them in this notebook.


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, display

ROOT = Path('.').resolve()
OUT_A = ROOT / 'outputs' / 'q1a'
OUT_B = ROOT / 'outputs' / 'q1b'
RESULTS = ROOT / 'model_results'

# False = read last run (preferred for Git). True = call Q1a.py / Q1b.py.
REFIT = False

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 140)
print('ROOT =', ROOT)


---
# Part A – Q1a: observed Alberta exports, 2025

**Question.** Top ten destinations by domestic export value, and top five HS4 products in each.

**Data I used.** Statistics Canada CIMT domestic exports 2025 (goods grown, extracted, or manufactured in Canada – not re-exports). Large monthly CSVs are not in Git; you can download them by running `Q1a.py`.

**Rules I applied.** Keep province `AB`; sum January–December 2025; roll US states up to the United States; roll HS6 to HS4; check Alberta totals match across HS2 / HS6 / HS8; attach public HS4 names.


In [ ]:
if REFIT:
    import Q1a
    Q1a.main()

top10_obs = pd.read_csv(OUT_A / 'q1a_top10_destinations.csv')
top5_obs = pd.read_csv(OUT_A / 'q1a_top5_hs4_by_destination.csv', dtype={'hs4_code': str})

total = top10_obs['domestic_export_value_cad'].sum() / top10_obs['share_of_alberta'].sum()
print(f'Alberta domestic exports (implied from top-10 shares): about CAD ${total:,.0f}')
print(f'Top 10 share of Alberta: {top10_obs["share_of_alberta"].sum():.1%}\n')

view = top10_obs.copy()
view['value'] = view['domestic_export_value_cad'].map(lambda v: f'${v:,.0f}')
view['share'] = view['share_of_alberta'].map(lambda s: f'{s:.1%}')
display(view[['destination_rank', 'destination', 'value', 'share']].rename(columns={
    'destination_rank': 'Rank', 'destination': 'Destination', 'value': 'CAD', 'share': 'Share'
}))


In [ ]:
fig_path = OUT_A / 'q1a_top10_destinations.png'
if fig_path.exists():
    display(Image(filename=str(fig_path)))

print('Top 5 HS4 products for each of the top 10 destinations:')
for dest, g in top5_obs.groupby('destination', sort=False):
    print(f'\n{dest}')
    for _, r in g.sort_values('product_rank').iterrows():
        name = str(r['product_name'])[:80]
        print(f"  {int(r['product_rank'])}. HS{r['hs4_code']} – {name} – ${r['domestic_export_value_cad']:,.0f}")


---
# Part B – Q1b: gravity-ML and Alberta export potential

**Question.** Model seven-economy HS4 trade; score Alberta as its own exporter; report potential top destinations and products.

**Training sample I used.** 2024 HS4 flows among seven economies (USA, Canada, Mexico, China, Korea, Japan, UK).

**Target.** `ln(1 + value USD)` so oil lines do not dominate every split.

**Validation.** 75/25 holdout by corridor (`GroupShuffleSplit`, `random_state=42`) and 5-fold `GroupKFold` by corridor. I emphasise ranking (top-5 HS4 hit, Spearman) as well as log R².


In [ ]:
if REFIT:
    import Q1b
    Q1b.main()

holdout = pd.read_csv(RESULTS / 'q1b_model_comparison.csv')
cv = pd.read_csv(RESULTS / 'q1b_cv5_mean_metrics.csv')
meta = json.loads((RESULTS / 'q1b_run_meta.json').read_text())

print('Scoring model for Alberta potential:', meta.get('scoring_display', meta.get('scoring_model')))
print('\nHoldout (pair split)')
cols = ['display_name', 'estimator', 'n_features', 'test_r2_ln', 'test_spearman_hs4', 'test_top5_hit', 'test_mdape']
h = holdout[cols].copy()
h = h.rename(columns={
    'display_name': 'Model',
    'estimator': 'Learner',
    'n_features': '# features',
    'test_r2_ln': 'R² (log)',
    'test_spearman_hs4': 'Spearman',
    'test_top5_hit': 'Top-5 hit',
    'test_mdape': 'MdAPE %',
})
display(h.sort_values('Top-5 hit', ascending=False))


In [ ]:
print('5-fold GroupKFold means')
c = cv[['display_name', 'r2_ln', 'spearman_hs4', 'top5_hit']].copy()
c = c.rename(columns={
    'display_name': 'Model',
    'r2_ln': 'CV R² (log)',
    'spearman_hs4': 'CV Spearman',
    'top5_hit': 'CV top-5 hit',
})
display(c.sort_values('CV top-5 hit', ascending=False))

for fig in ['q1b_model_comparison.png', 'q1b_residual_diagnostics.png']:
    p = OUT_B / fig
    if p.exists():
        print(fig)
        display(Image(filename=str(p)))


### How I read the ladder

- **Model 1** only knows size and distance. Every HS4 in a corridor gets the same prediction, so within-corridor Spearman is undefined and top-5 hit is about 0. It is the gravity floor.
- **Model 2** adds what is traded (chapter, energy/agri, RCA) and policy cost (MFN). Ranking becomes meaningful.
- **Model 3** adds how large and fast the destination market is for that product, how well export and import baskets match, and exporter capability. Holdout and CV ranking jump again.

Feature lists from the comparison file:


In [ ]:
for _, r in holdout.sort_values('model_id').iterrows():
    print(r['display_name'])
    print(' ', r['features'])
    print()


## Alberta as a separate exporter (Model 3)

- **Alberta-specific.** StatCan GDP and population (2023); distance from Calgary; contiguity only with the US.
- **Canadian proxies.** FTA, language, MFN facing Canadian origin.
- **Calibration.** One scale factor so predicted potential sums to Alberta’s observed 2025 domestic-export total. Ranks and product mix come from the model; the scale sets the level for presentation.


In [ ]:
top10_pot = pd.read_csv(OUT_B / 'q1b_predicted_top10_destinations.csv')
top5_pot = pd.read_csv(OUT_B / 'q1b_predicted_top5_hs4_by_destination.csv', dtype={'hs4': str})
compare = pd.read_csv(OUT_B / 'q1b_observed2025_vs_predicted_potential.csv')

print('Predicted Alberta export potential – top 10 destinations (Model 3)\n')
v = top10_pot.copy()
v['pred_usd'] = v['pred_value_usd'].map(lambda x: f'${x:,.0f}')
v['share'] = v['share_of_predicted_potential'].map(lambda s: f'{s:.1%}')
display(v[['rank', 'country_name', 'pred_usd', 'share', 'scoring_model']].rename(columns={
    'rank': 'Rank', 'country_name': 'Destination', 'pred_usd': 'Potential (USD)',
    'share': 'Share', 'scoring_model': 'Model id'
}))

cal = meta.get('calibration', {})
print(f"Calibration scale: {cal.get('scale', float('nan')):.4f}")


In [ ]:
print('Top 5 HS4 products by predicted destination (Model 3)\n')
for imp, g in top5_pot.groupby('importer', sort=False):
    name = g['country_name'].iloc[0]
    print(f'{name} ({imp})')
    for _, r in g.sort_values('rank_in_destination').iterrows():
        desc = str(r['hs4_name'])[:75]
        print(f"  {int(r['rank_in_destination'])}. HS{r['hs4']} – {desc} – ${r['pred_value_usd']:,.0f}")
    print()


## Observed 2025 vs model potential

Countries in both lists sit in the middle. Destinations only in Q1a are strong today; destinations only in Q1b potential are model-suggested relative to the training gravity-ML structure.


In [ ]:
c = compare.copy()
c['observed_cad'] = c['domestic_export_value_cad'].map(lambda x: f'${x:,.0f}' if pd.notna(x) else '–')
c['potential_usd'] = c['pred_value_usd'].map(lambda x: f'${x:,.0f}' if pd.notna(x) else '–')
c['name'] = c['country_name'].fillna(c['destination'])
display(c[['rank', 'name', 'potential_usd', 'observed_cad', 'share_of_alberta']].sort_values(
    'rank', na_position='last'
).rename(columns={
    'rank': 'Potential rank', 'name': 'Country',
    'potential_usd': 'Q1b potential (USD)', 'observed_cad': 'Q1a observed (CAD)',
    'share_of_alberta': 'Q1a share'
}))


---
## Where I put supporting material

| Path | Contents |
| --- | --- |
| `Q1a.py` / `Q1b.py` | Full runnable scripts |
| `methodology/Q1b_methodology.md` | Data, features, validation, Alberta assumptions |
| `methodology/APPENDIX_other_models.md` | RF, CatBoost, ensemble, and other trials I did not keep |
| `DATA_TERMS.md` | Source links and citation notes |
| `model_results/` | Holdout + CV CSVs, `q1b_run_meta.json` |
| `outputs/q1a/`, `outputs/q1b/` | Tables and figures |
| `README.md` | Install / run / provenance |

You can reuse this ladder for other provinces or years with a rebuilt panel. Discarded-estimator notes are in `methodology/APPENDIX_other_models.md`.
